# 07 — 항원-항체 interface (1A14 직접 다운로드)

> 본문: [`07_interface.md`](07_interface.md) 와 **한 절씩 짝지어** 보세요.
> **전 셀 실행 10초** (실측)

**이 노트북은 도구를 직접 돌립니다.** RCSB 에서 1A14 를 **직접 내려받아** contact 을 계산하고(`my_run/`), 커밋된 결과와 대조해요.
각 절은 **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서예요.
내가 만든 결과는 `my_run/` 에 쌓이고, 저장소에 커밋된 `data/` 는 **대조군(레퍼런스)** 으로만 씁니다.
어느 단계를 건너뛰거나 실패해도 `resolve()` 가 `data/` 로 폴백해서 뒤 절이 계속 돌아가요.

## 0) 부트스트랩 — 저장소 클론 · 도구 설치 · 작업 폴더 이동

**Colab**: 이 셀을 그대로 실행하면 클론 → 챕터 폴더 이동 → 필요한 도구 설치까지 한 번에 끝나요.
**로컬**: 챕터 폴더 안에서 열었다면 클론 없이 진행됩니다.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "07_interface"
PIP_PKGS = "pandas matplotlib biopython requests"          # 이 챕터가 실제로 돌리는 도구 (pip 이름)
NEED_HMMER = False        # ANARCI 계열은 HMMER 의 hmmscan 실행파일이 필요해요 (pip 로는 안 깔려요)
PIN_TRANSFORMERS = None    # IgFold 체크포인트가 요구하는 transformers 버전(없으면 None)

import os, sys, subprocess, pathlib, shutil, importlib.util
IN_COLAB = "google.colab" in sys.modules

def _run(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)

_MARK = "antibody_viz.py"           # 이 파일이 있는 폴더가 가이드 루트

def _find_root():
    """가이드 루트를 찾는다 — 챕터 폴더 안, 루트, 클론된 저장소 어디서 열어도 동작."""
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "repo 루트를 못 찾았습니다. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

ADV_ROOT = ROOT.resolve()
os.chdir(ADV_ROOT / CHAPTER)        # 챕터 폴더로 이동 → data/·my_run/ 상대경로 동작
sys.path.insert(0, str(ADV_ROOT))   # antibody_viz import 보장
PY, SCRIPTS = sys.executable, ADV_ROOT / "scripts"

# --- 의존성 설치 -----------------------------------------------------------
_IMPORT = {"biopython": "Bio", "pyyaml": "yaml"}          # pip 이름 ≠ import 이름
def _have(pkg):
    mod = _IMPORT.get(pkg, pkg.split("==")[0])
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

if NEED_HMMER and shutil.which("hmmscan") is None:
    # ANARCI 는 내부적으로 hmmscan 을 호출해요. pip install anarci 만으로는 안 깔려요.
    if IN_COLAB:
        _run("apt-get -qq update")                       # 인덱스가 낡으면 install 이 404 로 죽는다
        _run("apt-get -qq install -y hmmer")             # ← ANARCI 가 부르는 hmmscan
    else:
        print("[!] hmmscan 이 없어요 → conda install -c bioconda hmmer  (또는 apt install hmmer)")

_miss = [p for p in PIP_PKGS.split() if not _have(p)]
if _miss:
    _run(f'"{sys.executable}" -m pip -q install ' + " ".join(_miss))

if "igfold" in PIP_PKGS and importlib.util.find_spec("pkg_resources") is None:
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 해요.
    _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
    importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨집니다.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


if PIN_TRANSFORMERS:
    # IgFold 체크포인트에는 옛 transformers 의 토크나이저 객체가 pickle 돼 있어요.
    # 최신 transformers(5.x) 로는 unpickle 이 실패해서(Trie/BasicTokenizer 없음) 버전을 맞춥니다.
    _ver = subprocess.run([sys.executable, "-c",
                           "import transformers;print(transformers.__version__)"],
                          capture_output=True, text=True).stdout.strip()
    if not _ver.startswith("4."):
        print(f"[transformers {_ver or 'none'} → {PIN_TRANSFORMERS}] IgFold 체크포인트 호환 버전으로 맞춥니다")
        _run(f'"{sys.executable}" -m pip -q install "transformers=={PIN_TRANSFORMERS}"')

# --- 내 산출물 폴더 & 폴백 규칙 --------------------------------------------
MYRUN = pathlib.Path("my_run"); MYRUN.mkdir(exist_ok=True)

def run_tool(args, timeout=1800):
    """도구를 서브프로세스로 실제 실행하고 출력을 셀에 그대로 보여줘요."""
    args = [str(a) for a in args]
    print("$", " ".join(args))
    try:
        p = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
    except Exception as e:
        print(f"[실행 실패] {type(e).__name__}: {e}\n→ 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
        return False
    out = ((p.stdout or "") + (p.stderr or "")).strip()
    print(out[-3000:] if out else "(출력 없음)")
    if p.returncode != 0:
        print(f"[실패] returncode={p.returncode} → 이 단계는 건너뛰고 레퍼런스(data/)로 이어갑니다")
    return p.returncode == 0

def resolve(name):
    """내가 방금 만든 my_run/ 결과를 우선 쓰고, 없으면 커밋된 data/ 로 폴백."""
    mine, ref = MYRUN/name, pathlib.Path("data")/name
    if mine.exists():
        print(f"[내 결과]   {mine}")
        return str(mine)
    print(f"[레퍼런스] {ref}   ← my_run 산출물이 없어 커밋본으로 이어갑니다")
    return str(ref)

print("ADV_ROOT :", ADV_ROOT)
print("작업 폴더 :", pathlib.Path.cwd(), "| Colab:", IN_COLAB)

## 1) 직접 실행 — 복합체 다운로드 + contact 계산 (본문 7.2)

```bash
python scripts/pdb_contacts.py --pdb 1A14 --outdir my_run/pdb                       # ① 사슬 목록 확인
python scripts/pdb_contacts.py --pdb 1A14 --chain1 H --chain2 N --cutoff 4.0 \
    --outdir my_run/pdb --out my_run/contacts_H_N.tsv                                # ② 항원-항체
```
① 을 먼저 돌려 **사슬 ID 를 눈으로 확인**하는 게 핵심이에요 — 1A14 의 항원(neuraminidase)은 chain **N** 이에요.
네트워크가 막히면 `--fallback-cif` 로 커밋된 `data/pdb/1A14.cif` 를 씁니다(오프라인 대비).

In [ ]:
FALLBACK = ["--fallback-cif", "data/pdb/1A14.cif"]   # 다운로드 실패 시 커밋본 사용

run_tool([PY, SCRIPTS/"pdb_contacts.py", "--pdb", "1A14", "--outdir", "my_run/pdb", *FALLBACK])
ok_hn = run_tool([PY, SCRIPTS/"pdb_contacts.py", "--pdb", "1A14",
                  "--chain1", "H", "--chain2", "N", "--cutoff", "4.0",
                  "--outdir", "my_run/pdb", "--out", "my_run/contacts_H_N.tsv", *FALLBACK])
ok_hl = run_tool([PY, SCRIPTS/"pdb_contacts.py", "--pdb", "1A14",
                  "--chain1", "H", "--chain2", "L", "--cutoff", "4.0",
                  "--outdir", "my_run/pdb", "--out", "my_run/contacts_H_L.tsv", *FALLBACK])

## 2) 내 결과 확인 — paratope · epitope (본문 7.3)

In [ ]:
import pandas as pd
def load_contacts(path):
    rows = []
    for line in open(path):
        if "atom_contacts=" in line:
            left, n = line.rstrip().split("atom_contacts=")
            a, b = left.rstrip("\t").split("\t")
            rows.append((a.strip(), b.strip(), int(n)))
    return pd.DataFrame(rows, columns=["paratope (H)", "epitope (N)", "atom_contacts"])

df = load_contacts(resolve("contacts_H_N.tsv")).sort_values("atom_contacts", ascending=False)
print(f"항원-항체 contact: {len(df)} residue pairs, 총 {df['atom_contacts'].sum()} atom contacts")
display(df.head(8).reset_index(drop=True))

hl = load_contacts(resolve("contacts_H_L.tsv"))
print(f"비교) H–L (VH/VL packing): {len(hl)} residue pairs — 같은 항체인데 '무엇 대 무엇'이냐로 결과가 완전히 달라져요.")

## 3) 내 결과 확인 — interface contact 그래프 (본문 7.3)

In [ ]:
from antibody_viz import interface_contacts
from IPython.display import Image
png = "my_run/07_interface_contacts.png"
interface_contacts(resolve("contacts_H_N.tsv"),
                   "Antibody(H)–Antigen(N) contacts — 1A14 (≤4 Å, 내 실행 결과)", png)
Image(png)

## 4) 레퍼런스 대조 — 커밋된 contact 결과와 같은가

In [ ]:
import pathlib
def pairs(path):
    out = {}
    for line in open(path):
        if "atom_contacts=" in line:
            left, n = line.rstrip().split("atom_contacts=")
            a, b = left.rstrip("\t").split("\t")
            out[(a.strip(), b.strip())] = int(n)
    return out

if pathlib.Path("my_run/contacts_H_N.tsv").exists():
    mine, ref = pairs("my_run/contacts_H_N.tsv"), pairs("data/contacts_H_N.tsv")
    print(f"내 결과 {len(mine)} pairs / 레퍼런스 {len(ref)} pairs | 완전 일치 = {mine == ref}")
    only_mine = set(mine) - set(ref); only_ref = set(ref) - set(mine)
    if only_mine or only_ref:
        print("내 결과에만:", sorted(only_mine)); print("레퍼런스에만:", sorted(only_ref))
    print("원자접촉 수 차이:", {k: (mine[k], ref[k]) for k in set(mine) & set(ref) if mine[k] != ref[k]} or "없음")
else:
    print("my_run contact 결과가 없어 대조를 건너뜁니다.")

## 5) 복합체 3D 렌더 — paratope / epitope (본문 7.3)

항원(베이지 표면) + 항체 H(하늘)/L(연두) cartoon + paratope(주황 스틱)·epitope(빨강 스틱).

> 4절과 마찬가지로 **PyMOL 은 pip 설치가 안 돼요** → 커밋된 렌더를 보여줍니다(로컬에 PyMOL 이 있으면 재렌더).

In [ ]:
import shutil, subprocess
from IPython.display import Image
png = "07_complex_3d.png"
if shutil.which("pymol"):
    try:
        subprocess.run(["pymol", "-cq", str(ADV_ROOT/"scripts"/"render_07_complex.pml")],
                       cwd=str(ADV_ROOT), check=True, capture_output=True, text=True, timeout=180)
        print("PyMOL 재렌더 완료 →", png)
    except Exception as e:
        print("PyMOL 실행 실패 → 커밋된 렌더 표시:", type(e).__name__)
else:
    print("PyMOL 없음(예: Colab) → 커밋된 렌더를 표시합니다:", png)
Image(png)

> 다음 → 본문 [08. developability](../08_developability/08_developability.md)